# 68) Two-Way ANOVA Nedir?
One-Way ANOVA'da (Konu 66) sadece **tek bir kategorik değişkene** göre gruplama yapıyorduk (örn: sadece "Kanal"). Two-Way ANOVA ise **iki 
kategorik değişkeni aynı anda** analiz eder — hem her değişkenin **ayrı ayrı** etkisini, hem de ikisinin **birlikte (etkileşim/interaction)** etkisini test eder.

## Örnek Senaryo
"Destek kanalı (Telefon/Canlı Sohbet/Email/Sosyal Medya) VE müşteri segmenti (Bireysel/Kurumsal), memnuniyet puanını nasıl etkiliyor?" — burada 2 faktör var: Kanal ve Segment.

## Üç Ayrı Hipotez (Two-Way ANOVA'da Aynı Anda Test Edilir)
1. **Ana Etki 1 (Main Effect - Kanal):** Kanal'ın kendisi, segment fark etmeksizin, memnuniyeti etkiliyor mu?
   - H0: Tüm kanalların ortalama memnuniyeti eşittir
2. **Ana Etki 2 (Main Effect - Segment):** Segment'in kendisi, kanal fark etmeksizin, memnuniyeti etkiliyor mu?
   - H0: Bireysel ve Kurumsal segmentlerin ortalama memnuniyeti eşittir
3. **Etkileşim Etkisi (Interaction Effect):** Kanal'ın etkisi, segmente göre DEĞİŞİYOR mu? (Bu, Two-Way ANOVA'nın en değerli/ayırt edici kısmıdır)
   - H0: Kanal ve Segment arasında etkileşim yoktur

## Etkileşim (Interaction) Kavramı — En Kritik Nokta
Etkileşim, "bir faktörün etkisinin, diğer faktörün seviyesine göre değişmesi" demektir. Somut örnek: Belki **Bireysel** müşteriler için 
Canlı Sohbet en iyi kanal, ama **Kurumsal** müşteriler için Telefon daha iyi olabilir. Bu durumda "Kanal"ın etkisi, "Segment"e göre **değişiyor** — işte buna etkileşim denir. Sadece iki ayrı One-Way ANOVA yapsaydık (biri Kanal için, biri Segment için), bu etkileşimi **asla göremezdik.**

## Python'da Kullanımı
Two-Way ANOVA, `scipy`de doğrudan yoktur — `statsmodels`'ın formül tabanlı 
(`ols` + `anova_lm`) yapısı kullanılır:
```python
import statsmodels.api as sm
from statsmodels.formula.api import ols

model = ols('Memnuniyet ~ C(Kanal) + C(Segment) + C(Kanal):C(Segment)', data=df).fit()
anova_tablosu = sm.stats.anova_lm(model, typ=2)
print(anova_tablosu)
```
Formüldeki `C()`, "bunu kategorik değişken olarak ele al" demek. 
`Kanal:Segment` kısmı, etkileşim terimini temsil eder.

## Sonuç Tablosunun Okunması
Çıkan ANOVA tablosunda 3 ayrı satır olur (Kanal, Segment, Kanal:Segment), her birinin kendi F-istatistiği ve p-değeri vardır — üçü de **bağımsız olarak** yorumlanır.

In [2]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.formula.api import ols

np.random.seed(42)

kanallar = ['Telefon', 'Canlı Sohbet', 'Email', 'Sosyal Medya']
segmentler = ['Bireysel', 'Kurumsal']

ortalamalar = {
    ('Telefon', 'Bireysel'): 70, ('Telefon', 'Kurumsal'): 78,
    ('Canlı Sohbet', 'Bireysel'): 85, ('Canlı Sohbet', 'Kurumsal'): 75,
    ('Email', 'Bireysel'): 65, ('Email', 'Kurumsal'): 70,
    ('Sosyal Medya', 'Bireysel'): 80, ('Sosyal Medya', 'Kurumsal'): 68,
}

veriler = []
for (kanal, segment), ort in ortalamalar.items():
    puanlar = np.random.normal(loc=ort, scale=7, size=20)
    for p in puanlar:
        veriler.append({'Kanal': kanal, 'Segment': segment, 'Memnuniyet': p})

df = pd.DataFrame(veriler)
df.head()

# H0-1: Kanal'ın ortalama memnuniyet üzerinde etkisi yoktur.
# H0-2: Segment'in ortalama memnuniyet üzerinde etkisi yoktur.
# H0-3: Kanal ve Segment arasında etkileşim (interaction) yoktur.

model = ols('Memnuniyet ~ C(Kanal) + C(Segment) + C(Kanal):C(Segment)', data=df).fit()
anova_tablosu = sm.stats.anova_lm(model, typ=2)
print(anova_tablosu)

                          sum_sq     df          F        PR(>F)
C(Kanal)             3035.796169    3.0  22.780795  3.101668e-12
C(Segment)            163.824212    1.0   3.688040  5.667644e-02
C(Kanal):C(Segment)  2863.936980    3.0  21.491153  1.169084e-11
Residual             6751.901072  152.0        NaN           NaN


### Sonuç
Destek kanalı ve müşteri segmentinin, memnuniyet puanı üzerindeki etkisini test etmek amacıyla Two-Way ANOVA uyguladık. Sonuçlara göre Kanal'ın ana etkisi istatistiksel olarak anlamlıdır (F=22.78, p<0.001), Segment'in ana etkisi ise sınırda anlamsız bulunmuştur (F=3.69, p=0.057). Ancak Kanal ve Segment arasındaki etkileşim etkisi son derece anlamlıdır (F=21.49, p<0.001) — bu, Segment'in memnuniyet üzerindeki etkisinin, hangi kanalda olunduğuna göre değiştiğini göstermektedir. Örneğin Canlı Sohbet ve Sosyal Medya kanallarında Bireysel müşteriler daha memnunken, Telefon ve Email kanallarında Kurumsal müşteriler daha memnundur.